# SpaceX Falcon 9 First-Stage Landing Prediction
**IBM Data Science Capstone — reproducible analysis notebook**

**Business question:** Can launch characteristics help estimate whether a Falcon 9 first stage will land successfully?

**Data source:** IBM Skills Network course datasets. This notebook downloads the prepared feature dataset when run in Jupyter/Colab. It computes the charts, SQL-style summaries, map, and classification metrics from the data instead of hard-coding scores.

**Important:** Run all cells before submission so the outputs and metrics reflect the dataset available to you. If your course has a specific dataset file, upload it and change `DATA_URL` below.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os, sqlite3, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
r = requests.get(DATA_URL, timeout=30)
r.raise_for_status()
open("dataset_part_2.csv", "wb").write(r.content)
df = pd.read_csv("dataset_part_2.csv")
print("Shape:", df.shape)
display(df.head())
print(df.info())


## 1. Data wrangling

In [ ]:
# Inspect missingness and duplicate rows
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_values"))
print("Duplicate rows:", df.duplicated().sum())
# Standardize column labels for reliable downstream use
df.columns = [c.strip() for c in df.columns]
# Convert the target to integer where present
target_candidates = ["Class", "class", "Outcome", "outcome"]
target = next((c for c in target_candidates if c in df.columns), None)
if target is None:
    raise ValueError(f"Could not find landing target column. Columns found: {df.columns.tolist()}")
df[target] = pd.to_numeric(df[target], errors="coerce")
df = df.dropna(subset=[target]).drop_duplicates()
df[target] = df[target].astype(int)
print("Clean shape:", df.shape, "| Target distribution:")
display(df[target].value_counts().rename(index={0:"Unsuccessful",1:"Successful"}).to_frame("count"))


## 2. Exploratory data analysis and visualizations

In [ ]:
numeric = df.select_dtypes(include=np.number)
display(numeric.describe().T)
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df, x=target, ax=ax)
ax.set_title("Landing outcome distribution"); ax.set_xlabel("Landing class (1 = success)")
plt.tight_layout(); plt.show()
# Explore launch-site, orbit, and payload features when available
for col in ["LaunchSite", "Orbit", "BoosterVersion", "LandingPad"]:
    if col in df.columns:
        plt.figure(figsize=(10,4))
        sns.countplot(data=df, x=col, hue=target, order=df[col].value_counts().index)
        plt.title(f"Landing outcomes by {col}"); plt.xticks(rotation=35, ha="right")
        plt.tight_layout(); plt.show()
payload_col = next((c for c in ["PayloadMass", "Payload Mass (kg)", "payload_mass_kg"] if c in df.columns), None)
if payload_col:
    plt.figure(figsize=(7,4))
    sns.boxplot(data=df, x=target, y=payload_col)
    plt.title("Payload mass versus landing outcome")
    plt.tight_layout(); plt.show()


## 3. SQL EDA (SQLite)

In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("launches", conn, index=False, if_exists="replace")
columns = pd.read_sql_query("PRAGMA table_info(launches)", conn)["name"].tolist()
print("SQL table columns:", columns)
# Overall outcome counts
display(pd.read_sql_query(f'SELECT "{target}" AS landing_class, COUNT(*) AS launches FROM launches GROUP BY "{target}" ORDER BY launches DESC', conn))
# Group-level counts for useful categorical fields
for col in ["LaunchSite", "Orbit", "LandingPad"]:
    if col in columns:
        q = f'SELECT "{col}" AS category, COUNT(*) AS launches, AVG("{target}") AS success_rate FROM launches GROUP BY "{col}" ORDER BY success_rate DESC'
        print(f"Summary by {col}")
        display(pd.read_sql_query(q, conn))
conn.close()


## 4. Interactive visual analytics and geospatial mapping

In [ ]:
# Plotly dashboard-style charts (works in notebook)
try:
    import plotly.express as px
    cat = next((c for c in ["LaunchSite", "Orbit", "BoosterVersion"] if c in df.columns), None)
    if cat:
        fig = px.histogram(df, x=cat, color=target, barmode="group",
                           title=f"Launch outcomes by {cat}")
        fig.show()
    if payload_col:
        fig = px.scatter(df, x=payload_col, y=target, color=target,
                         title="Payload mass vs landing outcome")
        fig.show()
except ImportError:
    print("Install plotly with: pip install plotly")
# Folium map uses known launch-site coordinates and is optional
try:
    import folium
    sites = [
        ("CCAFS SLC-40", 28.5618571, -80.577366),
        ("KSC LC-39A", 28.6080585, -80.6039558),
        ("VAFB SLC-4E", 34.632093, -120.610829)
    ]
    m = folium.Map(location=[30,-95], zoom_start=4)
    for name, lat, lon in sites:
        folium.Marker([lat,lon], tooltip=name, popup=name).add_to(m)
    display(m)
except ImportError:
    print("Install folium with: pip install folium")


## 5. Predictive analysis

In [ ]:
# Use numeric predictors; remove target and obvious non-feature text/identifier fields.
X = df.drop(columns=[target])
X = X.select_dtypes(include=np.number).replace([np.inf,-np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
y = df[target]
if X.shape[1] == 0:
    raise ValueError("No numeric predictor columns found; inspect dataset schema.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.nunique()==2 else None
)
models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]),
    "SVM": Pipeline([("scale", StandardScaler()), ("model", SVC())]),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5)
}
results=[]
fitted={}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred=model.predict(X_test)
    fitted[name]=model
    results.append({
        "Model":name,
        "Accuracy":accuracy_score(y_test,pred),
        "Precision":precision_score(y_test,pred,zero_division=0),
        "Recall":recall_score(y_test,pred,zero_division=0),
        "F1":f1_score(y_test,pred,zero_division=0)
    })
results_df=pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df.style.format({c:"{:.3f}" for c in ["Accuracy","Precision","Recall","F1"]}))
best_name=results_df.iloc[0]["Model"]
best_model=fitted[best_name]
best_pred=best_model.predict(X_test)
print("Best model by test F1:", best_name)
print(classification_report(y_test,best_pred,zero_division=0))
cm=confusion_matrix(y_test,best_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion matrix — {best_name}")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout(); plt.show()


## 6. Findings, limitations, and recommendations

Use the computed tables and plots above to write the final numerical findings. Do not copy example metrics from other projects.

- **Business implication:** Landing prediction can help a launch-services competitor assess reuse-related cost advantages, but it cannot by itself determine a bid price.
- **Model selection:** Compare precision, recall, and F1 alongside accuracy because the outcome classes may be imbalanced.
- **Limitations:** The dataset is relatively small; launch technology and operations change over time; historical associations do not prove causation; a random train/test split may not represent future launches.
- **Next steps:** Validate with a chronological holdout, tune hyperparameters, inspect feature importance, and monitor performance as new launches occur.

## Reproducibility checklist
1. Run all cells from top to bottom.
2. Confirm no red error messages remain.
3. Save the notebook with outputs.
4. Replace placeholder GitHub URL in the report/slides after you create and upload your repository.
